# FAQ Intent Classification - DistilBERT Fine-Tuning (Kaggle GPU)

**Project:** distilbert_mini_project

### Before you run
1. Kaggle **Settings -> Accelerator -> GPU**
2. Upload `dataset/processed/` files as a Kaggle Dataset (train.csv, val.csv, test.csv, label_maps.json)
3. Set `DATA_DIR` in the config cell to your Kaggle input path
4. After training: download `/kaggle/working/faq_intent_model/`

In [ ]:
# --- Config ---
DATA_DIR = "/kaggle/input/faq-intent-processed"  # change to your Kaggle dataset path
OUTPUT_DIR = "/kaggle/working/faq_intent_model"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
SEED = 42

NUM_EPOCHS = 25
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

TEXT_COL = "text"
LABEL_COL = "label_id"

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DATA_DIR = Path(DATA_DIR)
for name in ["train.csv", "val.csv", "test.csv", "label_maps.json"]:
    assert (DATA_DIR / name).exists(), f"Missing {name} in {DATA_DIR}"

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class IntentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def tokenize_df(df, tokenizer, max_length, text_col):
    return tokenizer(
        df[text_col].tolist(),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors=None,
    )


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float(accuracy_score(labels, preds))}


set_seed(SEED)

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

with open(DATA_DIR / "label_maps.json") as f:
    label_maps = json.load(f)

id2label = {int(k): v for k, v in label_maps["id2label"].items()}
label2id = {k: int(v) for k, v in label_maps["label2id"].items()}
num_labels = len(id2label)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Classes: {num_labels}")
print("Sample intents:", list(id2label.values())[:5], "...")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

datasets = {}
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    enc = tokenize_df(df, tokenizer, MAX_LENGTH, TEXT_COL)
    labels = df[LABEL_COL].astype(int).tolist()
    datasets[name] = IntentDataset(enc, labels)

print("Tokenized datasets ready.")

In [ ]:
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
val_metrics = trainer.evaluate()
print("Train loss:", round(train_result.training_loss, 4))
print("Val metrics:", val_metrics)

In [ ]:
# Test set evaluation
predictions = trainer.predict(datasets["test"])
test_preds = np.argmax(predictions.predictions, axis=-1)
test_labels = test_df[LABEL_COL].astype(int).values

test_acc = accuracy_score(test_labels, test_preds)
print(f"Test accuracy: {test_acc:.4f}")

target_names = [id2label[i] for i in range(num_labels)]
print("\nClassification report:\n")
print(classification_report(test_labels, test_preds, target_names=target_names, zero_division=0))

print("Confusion matrix shape:", confusion_matrix(test_labels, test_preds).shape)

In [ ]:
# Save model + metrics for download
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metrics_report = {
    "train_loss": float(train_result.training_loss),
    "val_accuracy": float(val_metrics.get("eval_accuracy", 0)),
    "val_loss": float(val_metrics.get("eval_loss", 0)),
    "test_accuracy": float(test_acc),
    "num_labels": num_labels,
    "model_name": MODEL_NAME,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
}
with open(Path(OUTPUT_DIR) / "training_metrics.json", "w") as f:
    json.dump(metrics_report, f, indent=2)

print("Saved model to:", OUTPUT_DIR)
print("Metrics:", metrics_report)